In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from google.cloud import bigquery

In [ ]:
# =============================================================
# 1. CẤU HÌNH HỆ THỐNG
# =============================================================
PROJECT_ID = "project-7f16ff1d-9026-4799-bfa"
BUCKET = "amazon-reviews-lakehouse-data"
DATASET_ID = "gold_zone"
TABLE_NAME = "product_features"

# Đường dẫn dữ liệu
MASTER_PATH = f"gs://{BUCKET}/silver-zone/silver-master-table"
TEMP_GCS_BUCKET = BUCKET

# Tham số thời gian
CURRENT_TS = 1704067200  # Mốc hiện tại: 01/2024
SECONDS_IN_30_DAYS = 30 * 86400

# Khởi tạo Spark
spark = DataprocSparkSession.builder.getOrCreate()

In [ ]:
# =============================================================
# 2. HÀM XỬ LÝ CHÍNH
# =============================================================
def build_gold_product_features():
    print(f" Bắt đầu quy trình ETL cho bảng: {TABLE_NAME}")

    # Đọc dữ liệu từ Silver Zone
    df_master = spark.read.parquet(MASTER_PATH)

    # ---------------------------------------------------------
    # BƯỚC 1: TÍNH CÁC CHỈ SỐ CƠ BẢN (Rating, Price, Life Cycle)
    # ---------------------------------------------------------
    print(" Bước 1: Tính toán Rating, Price và Product Life Cycle...")

    product_base = df_master.groupBy("parent_asin").agg(
        F.avg("price").alias("price"),
        F.avg("rating").alias("average_rating"),
        F.count("user_id").alias("rating_number"),
        F.stddev("rating").alias("rating_variance"),
        F.min("timestamp").alias("first_review_ts"),
        F.max("main_category").alias("category")
    )

    # Tính số ngày kể từ review đầu tiên đến thời điểm CURRENT_TS
    product_base = product_base.withColumn(
        "product_life_cycle_days",
        (F.lit(CURRENT_TS) - F.col("first_review_ts").cast("long")) / 86400
    ).drop("first_review_ts")

    # ---------------------------------------------------------
    # BƯỚC 2: TÍNH PRODUCT VELOCITY (Độ hot 30 ngày qua)
    # ---------------------------------------------------------
    print(" Bước 2: Tính toán Product Velocity...")

    thirty_days_ago_ts = CURRENT_TS - SECONDS_IN_30_DAYS

    velocity_df = df_master \
        .filter(F.col("timestamp").cast("long") >= thirty_days_ago_ts) \
        .groupBy("parent_asin") \
        .agg(F.count("user_id").alias("product_velocity"))

In [ ]:
# ---------------------------------------------------------
# BƯỚC 3: XÁC ĐỊNH MÙA CAO ĐIỂM (Seasonality)
 # ---------------------------------------------------------
print(" Bước 3: Tìm tháng có lượng review cao nhất...")

    # Lấy tháng từ timestamp và đếm số review mỗi tháng
monthly_counts = df_master \
   .withColumn("month", F.month(F.from_unixtime(F.col("timestamp").cast("long")))) \
    .groupBy("parent_asin", "month") \
    .count()

# Sử dụng Window Function để tìm tháng có 'count' lớn nhất
window_spec = Window.partitionBy("parent_asin").orderBy(F.col("count").desc())

seasonality_df = monthly_counts \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") == 1) \
    .select("parent_asin", F.col("month").alias("peak_season_month"))

In [ ]:
# ---------------------------------------------------------
#   BƯỚC 4: TỔNG HỢP VÀ LƯU TRỮ
    # ---------------------------------------------------------
print("🔗 Đang kết hợp các thành phần dữ liệu...")

final_df = product_base \
    .join(velocity_df, "parent_asin", "left") \
    .join(seasonality_df, "parent_asin", "left") \
    .fillna(0, subset=["product_velocity"]) # Nếu không có review gần đây, velocity = 0

# Ghi dữ liệu vào BigQuery
full_table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"
print(f" Đang lưu bảng vào BigQuery: {full_table_id}")

final_df.write.format("bigquery") \
    .option("table", full_table_id) \
    .option("temporaryGcsBucket", TEMP_GCS_BUCKET) \
    .mode("overwrite") \
    .save()

print("✨ HOÀN TẤT! Dữ liệu đã sẵn sàng.")
final_df.show(5)   

In [ ]:
# =============================================================
# 3. THỰC THI
# =============================================================
if __name__ == "__main__":
    # 1. Kiểm tra/Tạo Dataset (Giữ nguyên logic cũ của bạn)
    bq_client = bigquery.Client(project=PROJECT_ID)
    dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"

    try:
        bq_client.get_dataset(dataset_ref)
        print(f" Dataset '{DATASET_ID}' đã tồn tại.")
    except Exception:
        print(f" Tạo mới dataset '{DATASET_ID}'...")
        new_ds = bigquery.Dataset(dataset_ref)
        new_ds.location = "asia-southeast1"
        bq_client.create_dataset(new_ds)

    # 2. Chạy tiến trình xử lý
    build_gold_product_features()